## Example 1: Basic MCP Client Setup and Protocol Basics

In [ ]:
# Example 1: Basic MCP Client Setup and Protocol Basics
from typing import Any, Dict, List
import json

import mcp
print("🔌 MCP (Model Context Protocol) - Basic Client Setup")

# MCP Protocol Information
print("📋 MCP Protocol Overview:")
print("   • Protocol: Model Context Protocol")
print("   • Purpose: Connect AI applications to data sources")
print("   • Architecture: Client-Server with standardized messaging")
print("   • Transport: JSON-RPC over stdio, HTTP, or WebSocket")

# Basic MCP client structure demo
class MCPClientDemo:
    def __init__(self):
        self.connected = False

    def simulate_connection(self):
        print("\n🔗 Simulating MCP Client Connection:")
        connection_params = {
            "protocol_version": "1.0",
            "client_info": {"name": "demo-client", "version": "1.0.0"},
            "capabilities": {"tools": True, "resources": True}
        }
        print(f"   Connection established with params: {connection_params}")
        self.connected = True
        return {"status": "connected", "protocol": "1.0"}

demo_client = MCPClientDemo()
result = demo_client.simulate_connection()
print(f"✅ Connection result: {result}")

# The handshake the demo describes, built with mcp's own types: they validate and round-trip.
import mcp.types as types
init_params = types.InitializeRequestParams(
    protocolVersion=types.LATEST_PROTOCOL_VERSION,
    capabilities=types.ClientCapabilities(),
    clientInfo=types.Implementation(name="demo-client", version="1.0.0"),
)
wire = init_params.model_dump(by_alias=True, exclude_none=True)
print(f"✅ InitializeRequestParams: {wire}")
assert wire["clientInfo"] == {"name": "demo-client", "version": "1.0.0"} and wire["protocolVersion"] == types.LATEST_PROTOCOL_VERSION, wire
assert types.InitializeRequestParams.model_validate(wire) == init_params

## Example 2: MCP Server Tools and Resource Management

In [ ]:
# Example 2: MCP Server Tools and Resource Management
import json
from typing import Any, Dict, List

try:
    print("🛠️ MCP Server Tools and Resource Management")
    
    # MCP server demo with tools and resources
    class MCPServerDemo:
        def __init__(self):
            self.tools = {}
            self.resources = {}
            
        def register_tool(self, name: str, description: str):
            self.tools[name] = {"name": name, "description": description}
            
        def call_tool(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
            if name == "calculate":
                expr = arguments.get("expression", "2+2")
                result = eval(expr)  # Demo only - unsafe in production
                return {"result": f"Calculation: {expr} = {result}"}
            return {"error": "Tool not found"}
            
        def add_resource(self, uri: str, content: str):
            self.resources[uri] = {"uri": uri, "content": content}
    
    # Initialize server and add capabilities
    server = MCPServerDemo()
    server.register_tool("calculate", "Perform mathematical calculations")
    server.add_resource("config://app.json", '{"version": "1.0", "debug": true}')
    
    # Demonstrate tool execution
    calc_result = server.call_tool("calculate", {"expression": "15 * 4 + 8"})
    print(f"🔧 Tool execution: {calc_result}")
    print(f"📊 Server summary: {len(server.tools)} tools, {len(server.resources)} resources")
    
except Exception as e:
    print(f"⚠️ Demo error: {e}")
    raise AssertionError(
        f"mcp example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: MCP Prompt Templates and Sampling Integration

In [ ]:
# Example 3: MCP Prompt Templates and Sampling Integration
import json
from typing import Any, Dict, List
from datetime import datetime

try:
    print("📝 MCP Prompt Templates and Sampling Integration")
    
    # MCP prompt and sampling manager
    class MCPPromptManager:
        def __init__(self):
            self.prompts = {}
            
        def register_prompt(self, name: str, template: str, parameters: List[str]):
            self.prompts[name] = {
                "name": name, 
                "template": template,
                "parameters": parameters
            }
            
        def get_prompt(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
            if name not in self.prompts:
                return {"error": f"Prompt '{name}' not found"}
            
            template = self.prompts[name]["template"]
            try:
                filled_template = template.format(**arguments)
                return {"prompt": filled_template, "status": "success"}
            except KeyError as e:
                return {"error": f"Missing parameter: {e}"}
    
    # Initialize and test prompt manager
    prompt_manager = MCPPromptManager()
    prompt_manager.register_prompt(
        "code_review",
        "Review this {language} code: {code}. Focus on {focus}.",
        ["language", "code", "focus"]
    )
    
    result = prompt_manager.get_prompt("code_review", {
        "language": "Python", 
        "code": "def hello(): return 'world'",
        "focus": "best practices"
    })
    print(f"📄 Prompt result: {result}")
    
except Exception as e:
    print(f"⚠️ Demo error: {e}")
    raise AssertionError(
        f"mcp example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: MCP Transport Layers and Security

In [ ]:
# Example 4: MCP Transport Layers and Security
import json
import hashlib
import hmac
from typing import Any, Dict, List

try:
    print("🔒 MCP Transport Layers and Security")
    
    # MCP transport manager with security
    class MCPTransportManager:
        def __init__(self):
            self.transports = {}
            self.connections = {}
            
        def register_transport(self, name: str, transport_type: str, endpoint: str):
            self.transports[name] = {
                "type": transport_type,
                "endpoint": endpoint,
                "security": "tls"
            }
            
        def connect(self, transport_name: str) -> Dict[str, Any]:
            if transport_name not in self.transports:
                return {"error": "Transport not found"}
            
            connection_id = f"conn_{len(self.connections)}"
            self.connections[connection_id] = {
                "transport": transport_name,
                "status": "connected"
            }
            return {"connection_id": connection_id, "status": "connected"}
            
        def generate_signature(self, message: str, secret: str) -> str:
            return hmac.new(secret.encode(), message.encode(), hashlib.sha256).hexdigest()
    
    # Test transport manager
    transport_manager = MCPTransportManager()
    transport_manager.register_transport("secure_http", "https", "https://api.example.com/mcp")
    
    connection = transport_manager.connect("secure_http")
    signature = transport_manager.generate_signature("test message", "secret_key")
    
    print(f"🔌 Connection: {connection}")
    print(f"🔐 Message signature: {signature[:16]}...")
    
except Exception as e:
    print(f"⚠️ Demo error: {e}")
    raise AssertionError(
        f"mcp example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Comprehensive MCP Integration and Production Deployment

In [ ]:
# Example 5: Comprehensive MCP Integration and Production Deployment
import json
import logging
import time
from typing import Any, Dict, List

try:
    print("🏭 Comprehensive MCP Integration - Production Deployment")
    
    # Production MCP server
    class ProductionMCPServer:
        def __init__(self, name: str):
            self.name = name
            self.version = "1.0.0"
            self.tools = {}
            self.metrics = {"requests": 0, "uptime": time.time()}
            
        def initialize(self) -> Dict[str, Any]:
            # Register production tools
            self.tools["database_query"] = {"description": "Execute database queries"}
            self.tools["file_ops"] = {"description": "File system operations"}
            
            return {
                "server_info": {"name": self.name, "version": self.version},
                "capabilities": {"tools": True, "resources": True}
            }
            
        def execute_tool(self, tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
            self.metrics["requests"] += 1
            
            if tool_name == "database_query":
                return {"result": f"Query executed: {args.get('query', 'SELECT 1')}"}
            elif tool_name == "file_ops":
                return {"result": f"File operation on: {args.get('path', '/tmp')}"}
            return {"error": "Tool not found"}
            
        def get_health(self) -> Dict[str, Any]:
            uptime = time.time() - self.metrics["uptime"]
            return {"status": "healthy", "uptime": f"{uptime:.1f}s", "requests": self.metrics["requests"]}
    
    # Test production server
    server = ProductionMCPServer("production-server")
    init_result = server.initialize()
    
    tool_result = server.execute_tool("database_query", {"query": "SELECT * FROM users"})
    health = server.get_health()
    
    print(f"🔧 Server initialized: {init_result['server_info']}")
    print(f"⚡ Tool result: {tool_result}")
    print(f"🏥 Health: {health}")
    
except Exception as e:
    print(f"⚠️ Demo error: {e}")
    raise AssertionError(
        f"mcp example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e